In [5]:
import sys
import subprocess
import importlib.util

print("JUPYTER PYTHON:")
print(sys.executable)

print("\nPYTHON VERSION:")
print(sys.version)

print("\nUPSTOX SDK:")
print(importlib.util.find_spec("upstox_client"))

print("\nPIP LOCATION:")
subprocess.run([sys.executable, "-m", "pip", "show", "upstox-python-sdk"])

JUPYTER PYTHON:
C:\Users\adity\AppData\Local\Python\pythoncore-3.15-64\python.exe

PYTHON VERSION:
3.15.2 (tags/v3.15.2:5e58eaa, Oct 25 2026, 17:27:03) [MSC v.1951 64 bit (AMD64)]

UPSTOX SDK:
None

PIP LOCATION:


CompletedProcess(args=['C:\\Users\\adity\\AppData\\Local\\Python\\pythoncore-3.15-64\\python.exe', '-m', 'pip', 'show', 'upstox-python-sdk'], returncode=1)

In [1]:
import sys
import upstox_client
import streamlit

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Upstox SDK: SUCCESS")
print("Streamlit:", streamlit.__version__)

Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Executable: C:\Users\adity\Downloads\Historical data\BharatMarket_AI\.venv-live\Scripts\python.exe
Upstox SDK: SUCCESS
Streamlit: 1.67.0


In [2]:

import sys
import inspect
import pandas as pd
import upstox_client

PROJECT_DIR = r"C:\Users\adity\Downloads\Historical data\BharatMarket_AI"

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from upstox_integration.config import settings

print("=" * 60)
print("BHARATMARKET AI — UPSTOX V3 SDK INSPECTION")
print("=" * 60)

# 1. Environment
print("\nPython:", sys.version.split()[0])
print("SDK:", upstox_client.__file__)

# 2. Streamer class
streamer_class = upstox_client.MarketDataStreamerV3

print("\n--- Streamer methods ---")

for method in [
    "connect",
    "disconnect",
    "subscribe",
    "unsubscribe",
    "change_mode",
    "auto_reconnect"
]:
    fn = getattr(streamer_class, method, None)

    if fn:
        try:
            print(f"{method}: {inspect.signature(fn)}")
        except Exception:
            print(f"{method}: AVAILABLE")
    else:
        print(f"{method}: NOT AVAILABLE")

# 3. Event interface
print("\n--- Event registration methods ---")

for method in ["on", "off"]:
    print(method, hasattr(streamer_class, method))

# 4. Configuration
print("\n--- Project configuration ---")

print("Token configured:", bool(settings.UPSTOX_ACCESS_TOKEN))
print("Mapping file:", settings.MAPPING_FILE.exists())

mapping = pd.read_csv(settings.MAPPING_FILE)

print("Instrument count:", len(mapping))
display(mapping)

# 5. Directories
print("\n--- Live directories ---")

for folder in [
    settings.UPSTOX_DIR / "live",
    settings.UPSTOX_DIR / "live" / "ticks",
    settings.UPSTOX_DIR / "live" / "candles",
    settings.UPSTOX_DIR / "live" / "logs"
]:
    folder.mkdir(parents=True, exist_ok=True)
    print("READY:", folder)

print("\nSDK INSPECTION COMPLETED")


BHARATMARKET AI — UPSTOX V3 SDK INSPECTION

Python: 3.12.10
SDK: C:\Users\adity\Downloads\Historical data\BharatMarket_AI\.venv-live\Lib\site-packages\upstox_client\__init__.py

--- Streamer methods ---
connect: (self)
disconnect: (self)
subscribe: (self, instrumentKeys, mode)
unsubscribe: (self, instrumentKeys)
change_mode: (self, instrumentKeys, newMode)
auto_reconnect: (self, enable, interval=1, retry_count=5)

--- Event registration methods ---
on True
off False

--- Project configuration ---
Token configured: True
Mapping file: True
Instrument count: 10


,BharatMarket_Symbol,Upstox_Instrument_Key
0,NIFTY50,NSE_INDEX|Nifty 50
1,BANKNIFTY,NSE_INDEX|Nifty Bank
2,RELIANCE,NSE_EQ|INE002A01018
3,TCS,NSE_EQ|INE467B01029
4,HDFCBANK,NSE_EQ|INE040A01034
5,INFY,NSE_EQ|INE009A01021
6,ICICIBANK,NSE_EQ|INE090A01021
7,SBIN,NSE_EQ|INE062A01020
8,ITC,NSE_EQ|INE154A01025
9,LT,NSE_EQ|INE018A01030



--- Live directories ---
READY: C:\Users\adity\Downloads\Historical data\BharatMarket_AI\upstox_integration\live
READY: C:\Users\adity\Downloads\Historical data\BharatMarket_AI\upstox_integration\live\ticks
READY: C:\Users\adity\Downloads\Historical data\BharatMarket_AI\upstox_integration\live\candles
READY: C:\Users\adity\Downloads\Historical data\BharatMarket_AI\upstox_integration\live\logs

SDK INSPECTION COMPLETED


In [3]:

import time
import threading
import traceback
from datetime import datetime
from zoneinfo import ZoneInfo

import upstox_client
import pandas as pd

# Load project configuration
from upstox_integration.config import settings

# Load instrument mappings
mapping = pd.read_csv(settings.MAPPING_FILE)

instrument_keys = (
    mapping["Upstox_Instrument_Key"]
    .dropna()
    .astype(str)
    .tolist()
)

# Capture connection results
received_messages = []
connection_events = []
subscription_status = {"subscribed": False}

print("=" * 65)
print("BHARATMARKET AI — LIVE FEED CONNECTION TEST")
print("=" * 65)

print("Instruments:", len(instrument_keys))
print("Mode: full")
print("Starting time:", datetime.now(ZoneInfo("Asia/Kolkata")))

# Configure authenticated API client
configuration = upstox_client.Configuration()
configuration.access_token = settings.UPSTOX_ACCESS_TOKEN

api_client = upstox_client.ApiClient(configuration)

# Create streamer
streamer = upstox_client.MarketDataStreamerV3(api_client)

def on_open(*args):
    print("\n[CONNECTED] Upstox WebSocket connection established.")

    connection_events.append("OPEN")

    try:
        streamer.subscribe(instrument_keys, "full")
        subscription_status["subscribed"] = True

        print("[SUBSCRIBED] All 10 instruments requested in FULL mode.")

    except Exception as exc:
        print("[SUBSCRIPTION ERROR]", str(exc))
        connection_events.append("SUBSCRIPTION_ERROR")

def on_message(message, *args):
    received_messages.append(message)

    # Display only the first five messages to avoid flooding Jupyter
    if len(received_messages) <= 5:
        print(f"\n[MESSAGE {len(received_messages)}]")
        print("Python type:", type(message).__name__)
        print("Payload preview:", repr(message)[:1800])

def on_error(*args):
    error_text = " ".join(str(x) for x in args)
    print("\n[WEBSOCKET ERROR]", error_text[:1500])
    connection_events.append("ERROR")

def on_close(*args):
    print("\n[DISCONNECTED]", args)
    connection_events.append("CLOSE")

# Register event handlers
streamer.on("open", on_open)
streamer.on("message", on_message)
streamer.on("error", on_error)
streamer.on("close", on_close)

# Enable limited automatic reconnection
streamer.auto_reconnect(True, 3, 3)

# Automatically stop the test after 45 seconds
disconnect_timer = threading.Timer(45, streamer.disconnect)
disconnect_timer.daemon = True

print("\nConnecting...")

try:
    disconnect_timer.start()
    streamer.connect()
    disconnect_timer.join(timeout=55)

except Exception as exc:
    print("\n[CONNECTION FAILURE]", repr(exc))
    traceback.print_exc()

finally:
    if disconnect_timer.is_alive():
        disconnect_timer.cancel()

print("\n" + "=" * 65)
print("CONNECTION TEST SUMMARY")
print("=" * 65)

print("Connection events:", connection_events)
print("Subscription requested:", subscription_status["subscribed"])
print("Messages received:", len(received_messages))
print("Test completed:", datetime.now(ZoneInfo("Asia/Kolkata")))


BHARATMARKET AI — LIVE FEED CONNECTION TEST
Instruments: 10
Mode: full
Starting time: 2026-11-25 20:30:05.135248+05:30

Connecting...

[CONNECTED] Upstox WebSocket connection established.
[SUBSCRIBED] All 10 instruments requested in FULL mode.

[MESSAGE 1]
Python type: dict
Payload preview: {'type': 'market_info', 'currentTs': '1795618806697', 'marketInfo': {'segmentStatus': {'BCD_FO': 'NORMAL_CLOSE', 'BSE_INDEX': 'CLOSING_END', 'NSE_COM': 'NORMAL_OPEN', 'BSE_EQ': 'CLOSING_END', 'MCX_FO': 'NORMAL_OPEN', 'NCD_FO': 'NORMAL_CLOSE', 'MCX_INDEX': 'NORMAL_OPEN', 'NSE_EG': 'NORMAL_OPEN', 'NSE_EQ': 'CLOSING_END', 'BSE_FO': 'NORMAL_CLOSE', 'US_EQ': 'NORMAL_CLOSE', 'NSE_INDEX': 'CLOSING_END', 'NSE_FO': 'NORMAL_CLOSE'}}}

[MESSAGE 2]
Python type: dict
Payload preview: {'feeds': {'NSE_EQ|INE154A01025': {'fullFeed': {'marketFF': {'ltpc': {'ltp': 309.25, 'ltt': '1795602573025', 'ltq': '4', 'cp': 301.0}, 'marketLevel': {'bidAskQuote': [{'askQ': '3819', 'askP': 309.25}, {}, {}, {}, {}]}, 'optionGreeks

In [4]:

import pandas as pd
from datetime import datetime, timezone

# Reverse instrument mapping
symbol_lookup = dict(
    zip(
        mapping["Upstox_Instrument_Key"],
        mapping["BharatMarket_Symbol"]
    )
)

rows = []

for message in received_messages:
    if not isinstance(message, dict):
        continue

    feeds = message.get("feeds", {})

    for instrument_key, feed_data in feeds.items():
        full_feed = feed_data.get("fullFeed", {})

        # Equity and index feeds have different nested structures
        market_data = (
            full_feed.get("marketFF")
            or full_feed.get("indexFF")
            or {}
        )

        ltpc = market_data.get("ltpc", {})

        ohlc_list = market_data.get("marketOHLC", {}).get("ohlc", [])

        daily = next(
            (item for item in ohlc_list if item.get("interval") == "1d"),
            {}
        )

        rows.append({
            "Symbol": symbol_lookup.get(instrument_key, instrument_key),
            "Instrument_Key": instrument_key,
            "LTP": ltpc.get("ltp"),
            "Previous_Close": ltpc.get("cp"),
            "Day_Open": daily.get("open"),
            "Day_High": daily.get("high"),
            "Day_Low": daily.get("low"),
            "Day_Close": daily.get("close"),
            "Volume": daily.get("vol"),
            "Last_Trade_Time": ltpc.get("ltt"),
            "Feed_Type": (
                "INDEX" if "indexFF" in full_feed else "EQUITY"
            )
        })

snapshot_df = pd.DataFrame(rows)

if not snapshot_df.empty:
    snapshot_df = snapshot_df.drop_duplicates(
        subset=["Instrument_Key"],
        keep="last"
    )

    display(snapshot_df)

    print("\nUnique instruments received:", snapshot_df["Instrument_Key"].nunique())
    print("Expected instruments:", len(mapping))
    print("Missing instruments:")

    received = set(snapshot_df["Instrument_Key"])
    expected = set(mapping["Upstox_Instrument_Key"])

    print(expected - received)
else:
    print("No instrument feed snapshots found.")


,Symbol,Instrument_Key,LTP,Previous_Close,Day_Open,Day_High,Day_Low,Day_Close,Volume,Last_Trade_Time,Feed_Type
0,ITC,NSE_EQ|INE154A01025,309.25,301.00,301.10,313.05,300.50,309.25,41071647,1795602573025,EQUITY
1,SBIN,NSE_EQ|INE062A01020,1138.15,1132.65,1142.00,1153.80,1132.00,1138.15,11608572,1795602596981,EQUITY
2,RELIANCE,NSE_EQ|INE002A01018,1373.90,1351.00,1359.80,1394.20,1357.00,1373.90,18781717,1795602589560,EQUITY
3,HDFCBANK,NSE_EQ|INE040A01034,846.20,817.80,833.10,850.50,831.55,846.20,48187313,1795602558752,EQUITY
4,INFY,NSE_EQ|INE009A01021,1130.00,1103.40,1141.80,1151.90,1122.40,1130.00,14498633,1795602592059,EQUITY
5,TCS,NSE_EQ|INE467B01029,2517.30,2460.00,2510.00,2534.00,2487.00,2517.30,5001661,1795602553519,EQUITY
6,BANKNIFTY,NSE_INDEX|Nifty Bank,63703.25,62585.35,63408.60,64133.80,63247.90,63703.25,NaN,1795602600000,INDEX
7,LT,NSE_EQ|INE018A01030,4156.00,4099.90,4169.80,4189.00,4127.30,4156.00,1313101,1795602509247,EQUITY
8,NIFTY50,NSE_INDEX|Nifty 50,25819.25,25349.65,25658.35,25951.65,25619.00,25819.25,NaN,1795602600000,INDEX
9,ICICIBANK,NSE_EQ|INE090A01021,1640.00,1585.50,1617.00,1650.00,1617.00,1640.00,18982380,1795602555908,EQUITY



Unique instruments received: 10
Expected instruments: 10
Missing instruments:
set()


In [5]:

from pathlib import Path
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo

symbol_lookup = dict(
    zip(
        mapping["Upstox_Instrument_Key"],
        mapping["BharatMarket_Symbol"]
    )
)

rows = []

for message in received_messages:
    if not isinstance(message, dict):
        continue

    for instrument_key, feed_data in message.get("feeds", {}).items():
        full_feed = feed_data.get("fullFeed", {})

        market_data = (
            full_feed.get("marketFF")
            or full_feed.get("indexFF")
            or {}
        )

        ltpc = market_data.get("ltpc", {})

        ohlc_list = market_data.get("marketOHLC", {}).get("ohlc", [])

        daily = next(
            (x for x in ohlc_list if x.get("interval") == "1d"),
            {}
        )

        rows.append({
            "Symbol": symbol_lookup.get(instrument_key, instrument_key),
            "Instrument_Key": instrument_key,
            "LTP": ltpc.get("ltp"),
            "Previous_Close": ltpc.get("cp"),
            "Day_Open": daily.get("open"),
            "Day_High": daily.get("high"),
            "Day_Low": daily.get("low"),
            "Day_Close": daily.get("close"),
            "Volume": daily.get("vol"),
            "Last_Trade_Time": ltpc.get("ltt"),
            "Feed_Type": "INDEX" if "indexFF" in full_feed else "EQUITY"
        })

snapshot_df = pd.DataFrame(rows)

snapshot_df = snapshot_df.drop_duplicates(
    subset=["Instrument_Key"],
    keep="last"
)

numeric_columns = [
    "LTP", "Previous_Close", "Day_Open",
    "Day_High", "Day_Low", "Day_Close", "Volume"
]

for col in numeric_columns:
    snapshot_df[col] = pd.to_numeric(snapshot_df[col], errors="coerce")

# Convert last trade timestamps from milliseconds to IST
snapshot_df["Last_Trade_Time_IST"] = pd.to_datetime(
    pd.to_numeric(snapshot_df["Last_Trade_Time"], errors="coerce"),
    unit="ms",
    utc=True
).dt.tz_convert("Asia/Kolkata")

# Validation
expected_keys = set(mapping["Upstox_Instrument_Key"])
received_keys = set(snapshot_df["Instrument_Key"])

print("Expected instruments:", len(expected_keys))
print("Received instruments:", len(received_keys))
print("Missing:", expected_keys - received_keys)
print("Duplicate instruments:", snapshot_df["Instrument_Key"].duplicated().sum())

display(snapshot_df.sort_values("Symbol").reset_index(drop=True))

# Save snapshot separately from historical daily CSVs
snapshot_path = (
    settings.UPSTOX_DIR / "live" / "latest_snapshot.csv"
)

snapshot_df.to_csv(snapshot_path, index=False)

print("\nSnapshot saved:", snapshot_path)
print("Snapshot rows:", len(snapshot_df))


Expected instruments: 10
Received instruments: 10
Missing: set()
Duplicate instruments: 0


,Symbol,Instrument_Key,LTP,Previous_Close,Day_Open,Day_High,Day_Low,Day_Close,Volume,Last_Trade_Time,Feed_Type,Last_Trade_Time_IST
0,BANKNIFTY,NSE_INDEX|Nifty Bank,63703.25,62585.35,63408.60,64133.80,63247.90,63703.25,NaN,1795602600000,INDEX,2026-11-25 16:00:00.000+05:30
1,HDFCBANK,NSE_EQ|INE040A01034,846.20,817.80,833.10,850.50,831.55,846.20,48187313.0,1795602558752,EQUITY,2026-11-25 15:59:18.752+05:30
2,ICICIBANK,NSE_EQ|INE090A01021,1640.00,1585.50,1617.00,1650.00,1617.00,1640.00,18982380.0,1795602555908,EQUITY,2026-11-25 15:59:15.908+05:30
3,INFY,NSE_EQ|INE009A01021,1130.00,1103.40,1141.80,1151.90,1122.40,1130.00,14498633.0,1795602592059,EQUITY,2026-11-25 15:59:52.059+05:30
4,ITC,NSE_EQ|INE154A01025,309.25,301.00,301.10,313.05,300.50,309.25,41071647.0,1795602573025,EQUITY,2026-11-25 15:59:33.025+05:30
5,LT,NSE_EQ|INE018A01030,4156.00,4099.90,4169.80,4189.00,4127.30,4156.00,1313101.0,1795602509247,EQUITY,2026-11-25 15:58:29.247+05:30
6,NIFTY50,NSE_INDEX|Nifty 50,25819.25,25349.65,25658.35,25951.65,25619.00,25819.25,NaN,1795602600000,INDEX,2026-11-25 16:00:00.000+05:30
7,RELIANCE,NSE_EQ|INE002A01018,1373.90,1351.00,1359.80,1394.20,1357.00,1373.90,18781717.0,1795602589560,EQUITY,2026-11-25 15:59:49.560+05:30
8,SBIN,NSE_EQ|INE062A01020,1138.15,1132.65,1142.00,1153.80,1132.00,1138.15,11608572.0,1795602596981,EQUITY,2026-11-25 15:59:56.981+05:30
9,TCS,NSE_EQ|INE467B01029,2517.30,2460.00,2510.00,2534.00,2487.00,2517.30,5001661.0,1795602553519,EQUITY,2026-11-25 15:59:13.519+05:30



Snapshot saved: C:\Users\adity\Downloads\Historical data\BharatMarket_AI\upstox_integration\live\latest_snapshot.csv
Snapshot rows: 10


In [7]:

import csv
import json
import threading
import time
from pathlib import Path
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

import pandas as pd
import upstox_client

from upstox_integration.config import settings

# --------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------

LIVE_DIR = settings.UPSTOX_DIR / "live"
TICK_DIR = LIVE_DIR / "ticks"
LOG_DIR = LIVE_DIR / "logs"

TICK_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

IST = ZoneInfo("Asia/Kolkata")

mapping = pd.read_csv(settings.MAPPING_FILE)

symbol_lookup = dict(
    zip(
        mapping["Upstox_Instrument_Key"],
        mapping["BharatMarket_Symbol"]
    )
)

instrument_keys = list(symbol_lookup.keys())

# Bounded test: 90 seconds
TEST_DURATION = 300

# --------------------------------------------------
# 2. OUTPUT FILES
# --------------------------------------------------

session_id = datetime.now(IST).strftime("%Y%m%d_%H%M%S")

raw_file = TICK_DIR / f"raw_feed_{session_id}.jsonl"
normalized_file = TICK_DIR / f"normalized_ticks_{session_id}.csv"

CSV_COLUMNS = [
    "received_at_ist",
    "message_kind",
    "symbol",
    "instrument_key",
    "feed_type",
    "ltp",
    "ltq",
    "previous_close",
    "exchange_timestamp_ms",
    "exchange_timestamp_ist",
    "cumulative_volume",
    "daily_ohlc_volume",
    "average_traded_price",
    "day_open",
    "day_high",
    "day_low",
    "day_close",
    "one_min_open",
    "one_min_high",
    "one_min_low",
    "one_min_close",
    "one_min_volume"
]

write_lock = threading.Lock()
first_feed_message = {"seen": False}
message_count = {"total": 0}
row_count = {"total": 0}
event_log = []

# --------------------------------------------------
# 3. TIMESTAMP CONVERSION
# --------------------------------------------------

def to_ist(timestamp_ms):
    if timestamp_ms is None:
        return None

    try:
        timestamp = int(timestamp_ms) / 1000
        return datetime.fromtimestamp(
            timestamp,
            tz=timezone.utc
        ).astimezone(IST).isoformat()
    except (ValueError, TypeError, OverflowError):
        return None

# --------------------------------------------------
# 4. SAVE NORMALIZED FEED
# --------------------------------------------------

def save_feed_message(message):
    global message_count, row_count

    received_at = datetime.now(IST)

    feeds = message.get("feeds", {})

    if not feeds:
        return

    with write_lock:
        message_kind = (
            "initial_snapshot"
            if not first_feed_message["seen"]
            else "feed_update"
        )

        first_feed_message["seen"] = True

        # Preserve the original decoded message
        with open(raw_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(message, default=str) + "\n")

        for instrument_key, feed_data in feeds.items():

            full_feed = feed_data.get("fullFeed", {})

            market_data = (
                full_feed.get("marketFF")
                or full_feed.get("indexFF")
                or {}
            )

            ltpc = market_data.get("ltpc", {})

            ohlc_list = market_data.get(
                "marketOHLC", {}
            ).get("ohlc", [])

            daily = next(
                (
                    item for item in ohlc_list
                    if item.get("interval") == "1d"
                ),
                {}
            )

            one_min = next(
                (
                    item for item in ohlc_list
                    if item.get("interval") == "I1"
                ),
                {}
            )

            exchange_timestamp = (
                ltpc.get("ltt")
                or message.get("currentTs")
            )

            row = {
                "received_at_ist": received_at.isoformat(),
                "message_kind": message_kind,
                "symbol": symbol_lookup.get(
                    instrument_key, instrument_key
                ),
                "instrument_key": instrument_key,
                "feed_type": (
                    "INDEX"
                    if "indexFF" in full_feed
                    else "EQUITY"
                ),
                "ltp": ltpc.get("ltp"),
                "ltq": ltpc.get("ltq"),
                "previous_close": ltpc.get("cp"),
                "exchange_timestamp_ms": exchange_timestamp,
                "exchange_timestamp_ist": to_ist(
                    exchange_timestamp
                ),
                "cumulative_volume": market_data.get("vtt"),
                "daily_ohlc_volume": daily.get("vol"),
                "average_traded_price": market_data.get("atp"),
                "day_open": daily.get("open"),
                "day_high": daily.get("high"),
                "day_low": daily.get("low"),
                "day_close": daily.get("close"),
                "one_min_open": one_min.get("open"),
                "one_min_high": one_min.get("high"),
                "one_min_low": one_min.get("low"),
                "one_min_close": one_min.get("close"),
                "one_min_volume": one_min.get("vol")
            }

            file_exists = normalized_file.exists()

            with open(
                normalized_file,
                "a",
                newline="",
                encoding="utf-8"
            ) as f:

                writer = csv.DictWriter(
                    f,
                    fieldnames=CSV_COLUMNS
                )

                if not file_exists:
                    writer.writeheader()

                writer.writerow(row)

            row_count["total"] += 1

        message_count["total"] += 1

# --------------------------------------------------
# 5. CREATE UPSTOX CONNECTION
# --------------------------------------------------

configuration = upstox_client.Configuration()
configuration.access_token = settings.UPSTOX_ACCESS_TOKEN

api_client = upstox_client.ApiClient(configuration)

collector = upstox_client.MarketDataStreamerV3(api_client)

def on_open(*args):
    print("[CONNECTED] WebSocket established.")
    event_log.append("OPEN")

    try:
        collector.subscribe(instrument_keys, "full")
        print("[SUBSCRIBED] 10 instruments in FULL mode.")
        event_log.append("SUBSCRIBED")
    except Exception as exc:
        print("[SUBSCRIPTION ERROR]", repr(exc))
        event_log.append("SUBSCRIPTION_ERROR")

def on_message(message, *args):
    try:
        save_feed_message(message)
    except Exception as exc:
        print("[DATA WRITE ERROR]", repr(exc))
        event_log.append("WRITE_ERROR")

def on_error(*args):
    print("[WEBSOCKET ERROR]", str(args)[:1000])
    event_log.append("ERROR")

def on_close(*args):
    print("[CLOSED]", args)
    event_log.append("CLOSE")

collector.on("open", on_open)
collector.on("message", on_message)
collector.on("error", on_error)
collector.on("close", on_close)

collector.auto_reconnect(True, 3, 3)

# --------------------------------------------------
# 6. RUN BOUNDED COLLECTION
# --------------------------------------------------

print("=" * 60)
print("BHARATMARKET AI — TICK COLLECTOR")
print("=" * 60)

print("Instruments:", len(instrument_keys))
print("Duration:", TEST_DURATION, "seconds")
print("Raw output:", raw_file)
print("Normalized output:", normalized_file)

disconnect_timer = threading.Timer(
    TEST_DURATION,
    collector.disconnect
)

disconnect_timer.daemon = True

try:
    disconnect_timer.start()
    collector.connect()
    disconnect_timer.join(timeout=TEST_DURATION + 15)

except Exception as exc:
    print("[COLLECTOR ERROR]", repr(exc))

finally:
    if disconnect_timer.is_alive():
        disconnect_timer.cancel()

# --------------------------------------------------
# 7. VALIDATE SAVED FILES
# --------------------------------------------------

print("\n" + "=" * 60)
print("COLLECTOR SUMMARY")
print("=" * 60)

print("Events:", event_log)
print("Feed messages saved:", message_count["total"])
print("Normalized instrument rows:", row_count["total"])
print("Raw file exists:", raw_file.exists())
print("Normalized file exists:", normalized_file.exists())

if normalized_file.exists():
    collected_df = pd.read_csv(normalized_file)

    print("CSV rows:", len(collected_df))
    print(
        "Unique instruments:",
        collected_df["instrument_key"].nunique()
    )

    display(
        collected_df[
            [
                "symbol",
                "ltp",
                "exchange_timestamp_ist",
                "cumulative_volume",
                "message_kind"
            ]
        ].head(20)
    )

print("\nTICK COLLECTION TEST COMPLETED")


BHARATMARKET AI — TICK COLLECTOR
Instruments: 10
Duration: 300 seconds
Raw output: C:\Users\adity\Downloads\Historical data\BharatMarket_AI\upstox_integration\live\ticks\raw_feed_20261125_204616.jsonl
Normalized output: C:\Users\adity\Downloads\Historical data\BharatMarket_AI\upstox_integration\live\ticks\normalized_ticks_20261125_204616.csv
[CONNECTED] WebSocket established.
[SUBSCRIBED] 10 instruments in FULL mode.
[CLOSED] (1000, None)

COLLECTOR SUMMARY
Events: ['OPEN', 'SUBSCRIBED', 'CLOSE']
Feed messages saved: 1
Normalized instrument rows: 10
Raw file exists: True
Normalized file exists: True
CSV rows: 10
Unique instruments: 10


,symbol,ltp,exchange_timestamp_ist,cumulative_volume,message_kind
0,ITC,309.25,2026-11-25T15:59:33.025000+05:30,41071647.0,initial_snapshot
1,SBIN,1138.15,2026-11-25T15:59:56.981000+05:30,11608572.0,initial_snapshot
2,RELIANCE,1373.90,2026-11-25T15:59:49.560000+05:30,18781717.0,initial_snapshot
3,HDFCBANK,846.20,2026-11-25T15:59:18.752000+05:30,48187313.0,initial_snapshot
4,INFY,1130.00,2026-11-25T15:59:52.059000+05:30,14498633.0,initial_snapshot
5,TCS,2517.30,2026-11-25T15:59:13.519000+05:30,5001661.0,initial_snapshot
6,BANKNIFTY,63703.25,2026-11-25T16:00:00+05:30,NaN,initial_snapshot
7,LT,4156.00,2026-11-25T15:58:29.247000+05:30,1313101.0,initial_snapshot
8,NIFTY50,25819.25,2026-11-25T16:00:00+05:30,NaN,initial_snapshot
9,ICICIBANK,1640.00,2026-11-25T15:59:15.908000+05:30,18982380.0,initial_snapshot



TICK COLLECTION TEST COMPLETED


In [8]:
import pandas as pd

df = pd.read_csv(normalized_file)

summary = (
    df.groupby("symbol")
    .agg(
        Total_Updates=("ltp", "count"),
        Unique_Prices=("ltp", "nunique"),
        First_Update=("received_at_ist", "min"),
        Last_Update=("received_at_ist", "max"),
        Missing_LTP=("ltp", lambda x: x.isna().sum())
    )
    .reset_index()
)

display(summary)

print("Total rows:", len(df))
print("Unique instruments:", df["symbol"].nunique())
print("Feed update rows:", (df["message_kind"] == "feed_update").sum())

,symbol,Total_Updates,Unique_Prices,First_Update,Last_Update,Missing_LTP
0,BANKNIFTY,1,1,2026-11-25T20:46:16.539696+05:30,2026-11-25T20:46:16.539696+05:30,0
1,HDFCBANK,1,1,2026-11-25T20:46:16.539696+05:30,2026-11-25T20:46:16.539696+05:30,0
2,ICICIBANK,1,1,2026-11-25T20:46:16.539696+05:30,2026-11-25T20:46:16.539696+05:30,0
3,INFY,1,1,2026-11-25T20:46:16.539696+05:30,2026-11-25T20:46:16.539696+05:30,0
4,ITC,1,1,2026-11-25T20:46:16.539696+05:30,2026-11-25T20:46:16.539696+05:30,0
5,LT,1,1,2026-11-25T20:46:16.539696+05:30,2026-11-25T20:46:16.539696+05:30,0
6,NIFTY50,1,1,2026-11-25T20:46:16.539696+05:30,2026-11-25T20:46:16.539696+05:30,0
7,RELIANCE,1,1,2026-11-25T20:46:16.539696+05:30,2026-11-25T20:46:16.539696+05:30,0
8,SBIN,1,1,2026-11-25T20:46:16.539696+05:30,2026-11-25T20:46:16.539696+05:30,0
9,TCS,1,1,2026-11-25T20:46:16.539696+05:30,2026-11-25T20:46:16.539696+05:30,0


Total rows: 10
Unique instruments: 10
Feed update rows: 0


In [9]:

from pathlib import Path
import pandas as pd
import numpy as np

from datetime import datetime
from zoneinfo import ZoneInfo

from upstox_integration.config import settings

LIVE_DIR = settings.UPSTOX_DIR / "live"
TICK_DIR = LIVE_DIR / "ticks"
CANDLE_DIR = LIVE_DIR / "candles"

CANDLE_DIR.mkdir(parents=True, exist_ok=True)

IST = "Asia/Kolkata"


def load_normalized_ticks(file_path):
    """Load and validate normalized Upstox feed records."""

    df = pd.read_csv(file_path)

    required = [
        "symbol",
        "instrument_key",
        "ltp",
        "exchange_timestamp_ms",
        "message_kind",
        "cumulative_volume"
    ]

    missing = set(required) - set(df.columns)

    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df["ltp"] = pd.to_numeric(df["ltp"], errors="coerce")

    df["exchange_timestamp_ms"] = pd.to_numeric(
        df["exchange_timestamp_ms"],
        errors="coerce"
    )

    df["cumulative_volume"] = pd.to_numeric(
        df["cumulative_volume"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["symbol", "ltp", "exchange_timestamp_ms"]
    ).copy()

    df["exchange_time"] = (
        pd.to_datetime(
            df["exchange_timestamp_ms"],
            unit="ms",
            utc=True
        )
        .dt.tz_convert(IST)
    )

    return df


def prepare_tick_updates(df):
    """
    Calculate volume increments from cumulative traded volume.

    Initial snapshot is used as a volume baseline.
    Only subsequent feed_update records contribute prices
    to candle construction.
    """

    df = df.sort_values(
        ["symbol", "exchange_time"]
    ).copy()

    # Remove exact repeated observations
    df = df.drop_duplicates(
        subset=[
            "instrument_key",
            "exchange_timestamp_ms",
            "ltp"
        ],
        keep="last"
    )

    # Calculate cumulative-volume increments
    df["volume_delta"] = (
        df.groupby("instrument_key")["cumulative_volume"]
        .diff()
    )

    # Do not treat negative volume changes as traded volume.
    # A reset is left unavailable for review.
    df.loc[df["volume_delta"] < 0, "volume_delta"] = np.nan

    # Initial observation has no measurable volume increment
    updates = df[
        df["message_kind"] == "feed_update"
    ].copy()

    updates = updates.dropna(
        subset=["exchange_time", "ltp"]
    )

    return updates


def build_candles(ticks, timeframe):
    """Aggregate tick observations into OHLC candles."""

    results = []

    for symbol, group in ticks.groupby("symbol"):

        group = group.sort_values("exchange_time").copy()

        group = group.set_index("exchange_time")

        # Equity volume is derived from cumulative-volume changes.
        # Index volume remains NaN.
        is_index = (
            group["instrument_key"]
            .astype(str)
            .str.startswith("NSE_INDEX|")
            .all()
        )

        if is_index:
            group["volume_delta"] = np.nan

        candles = group.resample(
            timeframe,
            label="left",
            closed="left"
        ).agg(
            open=("ltp", "first"),
            high=("ltp", "max"),
            low=("ltp", "min"),
            close=("ltp", "last"),
            tick_count=("ltp", "count"),
            volume=("volume_delta", "sum"),
        )

        candles = candles.dropna(subset=["open", "close"])

        if candles.empty:
            continue

        candles["symbol"] = symbol
        candles["timeframe"] = timeframe
        candles["candle_start"] = candles.index
        candles["candle_end"] = (
            candles.index + pd.Timedelta(timeframe)
        )

        candles["volume"] = (
            np.nan if is_index else candles["volume"]
        )

        candles = candles.reset_index(drop=True)

        results.append(candles)

    if not results:
        return pd.DataFrame()

    return pd.concat(results, ignore_index=True)


def run_candle_engine(file_path):
    """Build 1m, 5m and 15m candles from a tick file."""

    raw = load_normalized_ticks(file_path)

    updates = prepare_tick_updates(raw)

    print("Total normalized records:", len(raw))
    print("Initial snapshots:", (raw["message_kind"] == "initial_snapshot").sum())
    print("Actual feed updates:", len(updates))
    print("Instruments with updates:", updates["symbol"].nunique())

    if updates.empty:
        print("\nNo actual feed updates available.")
        print("Run the collector during NSE market hours.")
        return {}

    output = {}

    for timeframe, label in [
        ("1min", "1m"),
        ("5min", "5m"),
        ("15min", "15m")
    ]:

        candles = build_candles(updates, timeframe)

        if candles.empty:
            print(f"No candles generated for {label}.")
            continue

        filename = (
            CANDLE_DIR
            / f"intraday_candles_{label}.csv"
        )

        candles.to_csv(filename, index=False)

        output[label] = candles

        print(
            f"{label}: {len(candles)} candles | "
            f"{candles['symbol'].nunique()} instruments"
        )

        print("Saved:", filename)

    return output


print("CANDLE ENGINE READY")
print("Tick directory:", TICK_DIR)
print("Candle directory:", CANDLE_DIR)


CANDLE ENGINE READY
Tick directory: C:\Users\adity\Downloads\Historical data\BharatMarket_AI\upstox_integration\live\ticks
Candle directory: C:\Users\adity\Downloads\Historical data\BharatMarket_AI\upstox_integration\live\candles


In [10]:

from pathlib import Path

test_file = (
    TICK_DIR
    / "normalized_ticks_20261125_204255.csv"
)

if not test_file.exists():
    raise FileNotFoundError(test_file)

candle_results = run_candle_engine(test_file)


Total normalized records: 10
Initial snapshots: 10
Actual feed updates: 0
Instruments with updates: 0

No actual feed updates available.
Run the collector during NSE market hours.


In [11]:

from datetime import timedelta

base_time = pd.Timestamp(
    "2026-11-26 09:20:00",
    tz="Asia/Kolkata"
)

synthetic_ticks = pd.DataFrame({
    "symbol": ["TEST"] * 5,
    "instrument_key": ["TEST_KEY"] * 5,
    "exchange_time": [
        base_time,
        base_time + timedelta(seconds=15),
        base_time + timedelta(seconds=35),
        base_time + timedelta(minutes=1, seconds=5),
        base_time + timedelta(minutes=1, seconds=40),
    ],
    "ltp": [100, 101, 99, 102, 103],
    "volume_delta": [np.nan, 10, 15, 20, 25]
})

test_candles = build_candles(
    synthetic_ticks,
    "1min"
)

display(test_candles)

assert len(test_candles) == 2
assert test_candles.iloc[0]["open"] == 100
assert test_candles.iloc[0]["high"] == 101
assert test_candles.iloc[0]["low"] == 99
assert test_candles.iloc[0]["close"] == 99

assert test_candles.iloc[1]["open"] == 102
assert test_candles.iloc[1]["high"] == 103
assert test_candles.iloc[1]["low"] == 102
assert test_candles.iloc[1]["close"] == 103

print("PASS: 1-minute OHLC aggregation")
print("PASS: Two separate candle intervals")
print("PASS: Price ordering and aggregation")


,open,high,low,close,tick_count,volume,symbol,timeframe,candle_start,candle_end
0,100,101,99,99,3,25.0,TEST,1min,2026-11-26 09:20:00+05:30,2026-11-26 09:21:00+05:30
1,102,103,102,103,2,45.0,TEST,1min,2026-11-26 09:21:00+05:30,2026-11-26 09:22:00+05:30


PASS: 1-minute OHLC aggregation
PASS: Two separate candle intervals
PASS: Price ordering and aggregation


In [12]:

import numpy as np
import pandas as pd
from pathlib import Path

def calculate_rsi(series, period=14):
    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(
        alpha=1 / period,
        min_periods=period,
        adjust=False
    ).mean()

    avg_loss = loss.ewm(
        alpha=1 / period,
        min_periods=period,
        adjust=False
    ).mean()

    rs = avg_gain / avg_loss

    rsi = 100 - (100 / (1 + rs))

    # Handle constant-price windows
    rsi = rsi.mask(
        (avg_gain == 0) & (avg_loss == 0),
        50
    )

    rsi = rsi.mask(
        (avg_loss == 0) & (avg_gain > 0),
        100
    )

    return rsi


def calculate_atr(df, period=14):
    previous_close = df["close"].shift(1)

    true_range = pd.concat(
        [
            df["high"] - df["low"],
            (df["high"] - previous_close).abs(),
            (df["low"] - previous_close).abs()
        ],
        axis=1
    ).max(axis=1)

    atr = true_range.ewm(
        alpha=1 / period,
        min_periods=period,
        adjust=False
    ).mean()

    return atr


def calculate_indicators(candles):
    """Calculate indicators independently per symbol and timeframe."""

    if candles.empty:
        return pd.DataFrame()

    required = {
        "symbol", "timeframe", "candle_start",
        "open", "high", "low", "close", "volume"
    }

    missing = required - set(candles.columns)

    if missing:
        raise ValueError(f"Missing candle columns: {missing}")

    outputs = []

    for (symbol, timeframe), group in candles.groupby(
        ["symbol", "timeframe"]
    ):

        df = group.copy()

        df["candle_start"] = pd.to_datetime(
            df["candle_start"]
        )

        df = df.sort_values("candle_start").reset_index(drop=True)

        close = df["close"]

        # Moving averages
        df["ema_9"] = close.ewm(
            span=9, adjust=False, min_periods=9
        ).mean()

        df["ema_21"] = close.ewm(
            span=21, adjust=False, min_periods=21
        ).mean()

        df["ema_50"] = close.ewm(
            span=50, adjust=False, min_periods=50
        ).mean()

        # RSI
        df["rsi_14"] = calculate_rsi(close, 14)

        # MACD
        ema_12 = close.ewm(
            span=12, adjust=False, min_periods=12
        ).mean()

        ema_26 = close.ewm(
            span=26, adjust=False, min_periods=26
        ).mean()

        df["macd"] = ema_12 - ema_26

        df["macd_signal"] = df["macd"].ewm(
            span=9, adjust=False, min_periods=9
        ).mean()

        df["macd_histogram"] = (
            df["macd"] - df["macd_signal"]
        )

        # ATR
        df["atr_14"] = calculate_atr(df, 14)

        df["atr_percentage"] = (
            df["atr_14"] / close.replace(0, np.nan)
        ) * 100

        # Bollinger Bands
        rolling_mean = close.rolling(20).mean()
        rolling_std = close.rolling(20).std(ddof=0)

        df["bb_middle"] = rolling_mean
        df["bb_upper"] = rolling_mean + 2 * rolling_std
        df["bb_lower"] = rolling_mean - 2 * rolling_std

        df["bb_width"] = (
            (df["bb_upper"] - df["bb_lower"])
            / rolling_mean.replace(0, np.nan)
        ) * 100

        df["bb_position"] = (
            (close - df["bb_lower"])
            / (df["bb_upper"] - df["bb_lower"]).replace(0, np.nan)
        )

        # Session VWAP: only meaningful when valid volume exists
        df["typical_price"] = (
            df["high"] + df["low"] + df["close"]
        ) / 3

        df["vwap"] = np.nan

        volume = pd.to_numeric(df["volume"], errors="coerce")

        valid_volume = (
            volume.notna()
            & (volume >= 0)
        )

        if valid_volume.any():
            session_key = df["candle_start"].dt.date

            weighted_price = (
                df["typical_price"] * volume.where(valid_volume)
            )

            cumulative_pv = weighted_price.groupby(
                session_key
            ).cumsum()

            cumulative_volume = volume.where(
                valid_volume
            ).groupby(session_key).cumsum()

            df["vwap"] = (
                cumulative_pv
                / cumulative_volume.replace(0, np.nan)
            )

            # Do not report VWAP where the volume was unavailable
            df.loc[~valid_volume, "vwap"] = np.nan

        df["symbol"] = symbol
        df["timeframe"] = timeframe

        outputs.append(df)

    return pd.concat(outputs, ignore_index=True)


print("Intraday indicator engine functions created.")


Intraday indicator engine functions created.


In [13]:

# Generate isolated synthetic test candles
# These must never be added to production market data.

rng = np.random.default_rng(42)

test_dates = pd.date_range(
    "2026-11-26 09:15",
    periods=100,
    freq="1min",
    tz="Asia/Kolkata"
)

test_close = 100 + np.cumsum(
    rng.normal(0.02, 0.25, 100)
)

test_data = pd.DataFrame({
    "symbol": "TEST",
    "timeframe": "1min",
    "candle_start": test_dates,
    "candle_end": test_dates + pd.Timedelta(minutes=1),
    "open": test_close - rng.uniform(0.05, 0.15, 100),
    "high": test_close + rng.uniform(0.05, 0.25, 100),
    "low": test_close - rng.uniform(0.05, 0.25, 100),
    "close": test_close,
    "volume": rng.integers(100, 1000, 100),
    "tick_count": rng.integers(2, 10, 100)
})

# Ensure valid OHLC relationships
test_data["high"] = test_data[
    ["open", "high", "close"]
].max(axis=1)

test_data["low"] = test_data[
    ["open", "low", "close"]
].min(axis=1)

indicator_test = calculate_indicators(test_data)

expected_columns = [
    "rsi_14",
    "macd",
    "macd_signal",
    "macd_histogram",
    "atr_14",
    "ema_9",
    "ema_21",
    "ema_50",
    "bb_upper",
    "bb_lower",
    "vwap"
]

print("Rows:", len(indicator_test))

for column in expected_columns:
    assert column in indicator_test.columns
    print(
        column,
        "PASS" if indicator_test[column].notna().any()
        else "NOT READY"
    )

display(indicator_test.tail(5)[
    [
        "candle_start",
        "close",
        "rsi_14",
        "macd",
        "macd_signal",
        "atr_14",
        "ema_9",
        "ema_21",
        "ema_50",
        "vwap"
    ]
])

print("\nINDICATOR ENGINE TEST COMPLETED")


Rows: 100
rsi_14 PASS
macd PASS
macd_signal PASS
macd_histogram PASS
atr_14 PASS
ema_9 PASS
ema_21 PASS
ema_50 PASS
bb_upper PASS
bb_lower PASS
vwap PASS


,candle_start,close,rsi_14,macd,macd_signal,atr_14,ema_9,ema_21,ema_50,vwap
95,2026-11-26 10:50:00+05:30,101.369673,36.028509,-0.034224,0.012828,0.358568,101.857403,101.920958,101.794264,101.255887
96,2026-11-26 10:51:00+05:30,101.058998,31.131495,-0.097674,-0.009273,0.365269,101.697722,101.842598,101.765430,101.255541
97,2026-11-26 10:52:00+05:30,100.829686,28.095970,-0.164565,-0.040331,0.365448,101.524115,101.750515,101.728734,101.254452
98,2026-11-26 10:53:00+05:30,100.949629,31.839669,-0.205528,-0.073371,0.363169,101.409218,101.677708,101.698181,101.252877
99,2026-11-26 10:54:00+05:30,100.743260,29.038286,-0.251743,-0.109045,0.368661,101.276026,101.592758,101.660733,101.243472



INDICATOR ENGINE TEST COMPLETED


In [14]:

from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\adity\Downloads\Historical data\BharatMarket_AI"
)

LIVE_DIR = (
    PROJECT_DIR
    / "upstox_integration"
    / "live"
)

LIVE_DIR.mkdir(parents=True, exist_ok=True)

indicator_code = r'''
import numpy as np
import pandas as pd


def calculate_rsi(series, period=14):
    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(
        alpha=1 / period,
        min_periods=period,
        adjust=False
    ).mean()

    avg_loss = loss.ewm(
        alpha=1 / period,
        min_periods=period,
        adjust=False
    ).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    rsi = rsi.mask(
        (avg_gain == 0) & (avg_loss == 0), 50
    )

    rsi = rsi.mask(
        (avg_loss == 0) & (avg_gain > 0), 100
    )

    return rsi


def calculate_atr(df, period=14):
    previous_close = df["close"].shift(1)

    true_range = pd.concat(
        [
            df["high"] - df["low"],
            (df["high"] - previous_close).abs(),
            (df["low"] - previous_close).abs()
        ],
        axis=1
    ).max(axis=1)

    return true_range.ewm(
        alpha=1 / period,
        min_periods=period,
        adjust=False
    ).mean()


def calculate_indicators(candles):

    if candles.empty:
        return pd.DataFrame()

    required = {
        "symbol", "timeframe", "candle_start",
        "open", "high", "low", "close", "volume"
    }

    missing = required - set(candles.columns)

    if missing:
        raise ValueError(
            f"Missing candle columns: {missing}"
        )

    outputs = []

    for (symbol, timeframe), group in candles.groupby(
        ["symbol", "timeframe"]
    ):

        df = group.copy()

        df["candle_start"] = pd.to_datetime(
            df["candle_start"]
        )

        df = df.sort_values(
            "candle_start"
        ).reset_index(drop=True)

        close = df["close"]

        # Exponential moving averages
        df["ema_9"] = close.ewm(
            span=9, adjust=False, min_periods=9
        ).mean()

        df["ema_21"] = close.ewm(
            span=21, adjust=False, min_periods=21
        ).mean()

        df["ema_50"] = close.ewm(
            span=50, adjust=False, min_periods=50
        ).mean()

        # RSI
        df["rsi_14"] = calculate_rsi(close, 14)

        # MACD
        ema_12 = close.ewm(
            span=12, adjust=False, min_periods=12
        ).mean()

        ema_26 = close.ewm(
            span=26, adjust=False, min_periods=26
        ).mean()

        df["macd"] = ema_12 - ema_26

        df["macd_signal"] = df["macd"].ewm(
            span=9, adjust=False, min_periods=9
        ).mean()

        df["macd_histogram"] = (
            df["macd"] - df["macd_signal"]
        )

        # ATR
        df["atr_14"] = calculate_atr(df, 14)

        df["atr_percentage"] = (
            df["atr_14"] / close.replace(0, np.nan)
        ) * 100

        # Bollinger Bands
        rolling_mean = close.rolling(20).mean()
        rolling_std = close.rolling(20).std(ddof=0)

        df["bb_middle"] = rolling_mean
        df["bb_upper"] = rolling_mean + 2 * rolling_std
        df["bb_lower"] = rolling_mean - 2 * rolling_std

        df["bb_width"] = (
            (df["bb_upper"] - df["bb_lower"])
            / rolling_mean.replace(0, np.nan)
        ) * 100

        df["bb_position"] = (
            (close - df["bb_lower"])
            / (df["bb_upper"] - df["bb_lower"]).replace(
                0, np.nan
            )
        )

        # Session VWAP
        df["typical_price"] = (
            df["high"] + df["low"] + df["close"]
        ) / 3

        df["vwap"] = np.nan

        volume = pd.to_numeric(
            df["volume"], errors="coerce"
        )

        valid_volume = volume.notna() & (volume >= 0)

        if valid_volume.any():

            session_key = df["candle_start"].dt.date

            weighted_price = (
                df["typical_price"]
                * volume.where(valid_volume)
            )

            cumulative_pv = weighted_price.groupby(
                session_key
            ).cumsum()

            cumulative_volume = volume.where(
                valid_volume
            ).groupby(session_key).cumsum()

            df["vwap"] = (
                cumulative_pv
                / cumulative_volume.replace(0, np.nan)
            )

            df.loc[~valid_volume, "vwap"] = np.nan

        df["symbol"] = symbol
        df["timeframe"] = timeframe

        outputs.append(df)

    return pd.concat(outputs, ignore_index=True)
'''

indicator_file = LIVE_DIR / "indicator_engine.py"

indicator_file.write_text(
    indicator_code,
    encoding="utf-8"
)

print("FILE CREATED SUCCESSFULLY")
print("Location:", indicator_file)
print("File size:", indicator_file.stat().st_size, "bytes")


FILE CREATED SUCCESSFULLY
Location: C:\Users\adity\Downloads\Historical data\BharatMarket_AI\upstox_integration\live\indicator_engine.py
File size: 4580 bytes


In [15]:
import sys
import importlib

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import upstox_integration.live.indicator_engine as indicator_engine

importlib.reload(indicator_engine)

print("Indicator engine imported successfully!")

print("Functions available:")
print([
    "calculate_rsi",
    "calculate_atr",
    "calculate_indicators"
])

print("File:", indicator_engine.__file__)

Indicator engine imported successfully!
Functions available:
['calculate_rsi', 'calculate_atr', 'calculate_indicators']
File: C:\Users\adity\Downloads\Historical data\BharatMarket_AI\upstox_integration\live\indicator_engine.py


In [16]:

import numpy as np
import pandas as pd

from upstox_integration.live.indicator_engine import (
    calculate_rsi,
    calculate_atr,
    calculate_indicators
)

# Generate isolated test data
rng = np.random.default_rng(42)

dates = pd.date_range(
    "2026-11-26 09:15",
    periods=120,
    freq="1min",
    tz="Asia/Kolkata"
)

close = 100 + np.cumsum(rng.normal(0.02, 0.25, 120))

test_candles = pd.DataFrame({
    "symbol": "TEST",
    "timeframe": "1min",
    "candle_start": dates,
    "open": close - 0.10,
    "high": close + 0.20,
    "low": close - 0.20,
    "close": close,
    "volume": rng.integers(100, 1000, 120)
})

indicators = calculate_indicators(test_candles)

required_indicators = [
    "rsi_14",
    "macd",
    "macd_signal",
    "macd_histogram",
    "atr_14",
    "atr_percentage",
    "ema_9",
    "ema_21",
    "ema_50",
    "bb_upper",
    "bb_lower",
    "bb_width",
    "bb_position",
    "vwap"
]

print("INDICATOR MODULE VALIDATION")
print("=" * 55)

for column in required_indicators:
    valid = (
        column in indicators.columns
        and indicators[column].notna().any()
    )

    print(
        f"{column:20} : {'PASS' if valid else 'FAIL'}"
    )

assert indicators["rsi_14"].dropna().between(0, 100).all()
assert (indicators["atr_14"].dropna() >= 0).all()

assert len(indicators) == 120

print("\nRows calculated:", len(indicators))
print("Indicators calculated:", len(required_indicators))

display(
    indicators.tail(5)[
        [
            "candle_start",
            "close",
            "rsi_14",
            "macd",
            "atr_14",
            "ema_9",
            "ema_21",
            "ema_50",
            "vwap"
        ]
    ]
)

print("\nINDICATOR MODULE TEST COMPLETED")


INDICATOR MODULE VALIDATION
rsi_14               : PASS
macd                 : PASS
macd_signal          : PASS
macd_histogram       : PASS
atr_14               : PASS
atr_percentage       : PASS
ema_9                : PASS
ema_21               : PASS
ema_50               : PASS
bb_upper             : PASS
bb_lower             : PASS
bb_width             : PASS
bb_position          : PASS
vwap                 : PASS

Rows calculated: 120
Indicators calculated: 14


,candle_start,close,rsi_14,macd,atr_14,ema_9,ema_21,ema_50,vwap
115,2026-11-26 11:10:00+05:30,100.103705,31.594612,-0.353521,0.434357,100.430961,100.768743,101.161944,101.200588
116,2026-11-26 11:11:00+05:30,100.181874,33.985482,-0.354672,0.431902,100.381144,100.715391,101.123510,101.186667
117,2026-11-26 11:12:00+05:30,100.207338,34.785082,-0.349500,0.429624,100.346382,100.669205,101.087581,101.172860
118,2026-11-26 11:13:00+05:30,100.627782,46.342200,-0.307925,0.443254,100.402662,100.665439,101.069550,101.170305
119,2026-11-26 11:14:00+05:30,100.587943,45.519050,-0.275022,0.440164,100.439719,100.658394,101.050663,101.162220



INDICATOR MODULE TEST COMPLETED


In [17]:

from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\adity\Downloads\Historical data\BharatMarket_AI"
)

LIVE_DIR = PROJECT_DIR / "upstox_integration" / "live"

pipeline_code = r'''
import numpy as np
import pandas as pd

from pathlib import Path
from zoneinfo import ZoneInfo

from upstox_integration.live.indicator_engine import (
    calculate_indicators
)

IST = "Asia/Kolkata"


def load_ticks(file_path):
    """Load and validate normalized Upstox observations."""

    df = pd.read_csv(file_path)

    required = [
        "symbol",
        "instrument_key",
        "ltp",
        "exchange_timestamp_ms",
        "message_kind",
        "cumulative_volume"
    ]

    missing = set(required) - set(df.columns)

    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df["ltp"] = pd.to_numeric(df["ltp"], errors="coerce")

    df["exchange_timestamp_ms"] = pd.to_numeric(
        df["exchange_timestamp_ms"], errors="coerce"
    )

    df["cumulative_volume"] = pd.to_numeric(
        df["cumulative_volume"], errors="coerce"
    )

    df = df.dropna(
        subset=["symbol", "instrument_key", "ltp",
                "exchange_timestamp_ms"]
    ).copy()

    df["exchange_time"] = pd.to_datetime(
        df["exchange_timestamp_ms"],
        unit="ms",
        utc=True
    ).dt.tz_convert(IST)

    df["session_date"] = df["exchange_time"].dt.date

    df = df.sort_values(
        ["instrument_key", "exchange_time"]
    )

    # Remove exact duplicate observations
    df = df.drop_duplicates(
        subset=[
            "instrument_key",
            "exchange_timestamp_ms",
            "ltp"
        ],
        keep="last"
    )

    return df


def prepare_live_updates(df):
    """
    Calculate observed cumulative-volume increments.

    Initial snapshots establish the cumulative-volume baseline.
    Only subsequent feed updates contribute candle prices.
    """

    df = df.copy()

    df["observed_volume_delta"] = (
        df.groupby(
            ["instrument_key", "session_date"]
        )["cumulative_volume"].diff()
    )

    # A negative increment may indicate a volume reset.
    # Do not convert it into positive or zero volume.
    df.loc[
        df["observed_volume_delta"] < 0,
        "observed_volume_delta"
    ] = np.nan

    updates = df[
        df["message_kind"] == "feed_update"
    ].copy()

    # NSE regular-session observations only
    session_time = updates["exchange_time"].dt.time

    start_time = pd.Timestamp("09:15").time()
    end_time = pd.Timestamp("15:30").time()

    updates = updates[
        (session_time >= start_time)
        & (session_time < end_time)
    ].copy()

    return updates


def build_candles(ticks, timeframe):
    """Build OHLC candles separately for each instrument and day."""

    if ticks.empty:
        return pd.DataFrame()

    results = []

    for (symbol, instrument_key, session_date), group in ticks.groupby(
        ["symbol", "instrument_key", "session_date"]
    ):

        group = group.sort_values("exchange_time").copy()

        group = group.set_index("exchange_time")

        is_index = str(instrument_key).startswith("NSE_INDEX|")

        # Index volume is unavailable.
        if is_index:
            group["observed_volume_delta"] = np.nan

        candles = group.resample(
            timeframe,
            origin="start_day",
            offset="15min",
            label="left",
            closed="left"
        ).agg(
            open=("ltp", "first"),
            high=("ltp", "max"),
            low=("ltp", "min"),
            close=("ltp", "last"),
            tick_count=("ltp", "count"),
            observed_volume=("observed_volume_delta",
                             lambda x: x.sum(min_count=1))
        )

        candles = candles.dropna(
            subset=["open", "high", "low", "close"]
        )

        if candles.empty:
            continue

        candles["symbol"] = symbol
        candles["instrument_key"] = instrument_key
        candles["session_date"] = str(session_date)
        candles["timeframe"] = timeframe

        candles["candle_start"] = candles.index
        candles["candle_end"] = (
            candles.index + pd.Timedelta(timeframe)
        )

        candles["is_index"] = is_index

        if is_index:
            candles["observed_volume"] = np.nan

        candles = candles.reset_index(drop=True)

        results.append(candles)

    if not results:
        return pd.DataFrame()

    return pd.concat(results, ignore_index=True)


def process_intraday_file(file_path, output_dir):
    """Create candles and technical indicators for one tick file."""

    file_path = Path(file_path)
    output_dir = Path(output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    ticks = load_ticks(file_path)
    updates = prepare_live_updates(ticks)

    print("Normalized observations:", len(ticks))
    print("Actual feed updates:", len(updates))
    print("Instruments with updates:", updates["symbol"].nunique())

    if updates.empty:
        print("No genuine intraday updates available.")
        return {}

    session_id = file_path.stem

    results = {}

    for rule, label in [
        ("1min", "1m"),
        ("5min", "5m"),
        ("15min", "15m")
    ]:

        candles = build_candles(updates, rule)

        if candles.empty:
            print(f"No {label} candles generated.")
            continue

        # A candle is complete only after its interval ends.
        now = pd.Timestamp.now(tz=IST)

        candles["is_complete"] = (
            pd.to_datetime(candles["candle_end"], utc=True)
            .dt.tz_convert(IST) <= now
        )

        # Indicators are calculated separately by instrument/timeframe.
        indicators = calculate_indicators(candles)

        candle_path = (
            output_dir
            / f"candles_{label}_{session_id}.csv"
        )

        indicator_path = (
            output_dir
            / f"indicators_{label}_{session_id}.csv"
        )

        candles.to_csv(candle_path, index=False)
        indicators.to_csv(indicator_path, index=False)

        results[label] = {
            "candles": candles,
            "indicators": indicators,
            "candle_file": candle_path,
            "indicator_file": indicator_path
        }

        print(
            f"{label}: {len(candles)} candles | "
            f"{candles['symbol'].nunique()} instruments"
        )

        print("Candles:", candle_path)
        print("Indicators:", indicator_path)

    return results
'''

pipeline_file = LIVE_DIR / "intraday_pipeline.py"

pipeline_file.write_text(
    pipeline_code,
    encoding="utf-8"
)

print("INTRADAY PIPELINE CREATED")
print("Location:", pipeline_file)
print("Size:", pipeline_file.stat().st_size, "bytes")


INTRADAY PIPELINE CREATED
Location: C:\Users\adity\Downloads\Historical data\BharatMarket_AI\upstox_integration\live\intraday_pipeline.py
Size: 6618 bytes


In [18]:

import sys
import importlib

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from upstox_integration.live import intraday_pipeline

importlib.reload(intraday_pipeline)

print("PIPELINE IMPORT SUCCESSFUL")

print("Functions available:")
print([
    "load_ticks",
    "prepare_live_updates",
    "build_candles",
    "process_intraday_file"
])

print("Module:", intraday_pipeline.__file__)


PIPELINE IMPORT SUCCESSFUL
Functions available:
['load_ticks', 'prepare_live_updates', 'build_candles', 'process_intraday_file']
Module: C:\Users\adity\Downloads\Historical data\BharatMarket_AI\upstox_integration\live\intraday_pipeline.py


In [19]:

from upstox_integration.live.intraday_pipeline import (
    process_intraday_file
)

test_file = (
    LIVE_DIR
    / "ticks"
    / "normalized_ticks_20261125_204255.csv"
)

output_dir = LIVE_DIR / "candles"

pipeline_results = process_intraday_file(
    test_file,
    output_dir
)


Normalized observations: 10
Actual feed updates: 0
Instruments with updates: 0
No genuine intraday updates available.


In [20]:

from pathlib import Path
from datetime import datetime
import shutil

# ==========================================
# BHARATMARKET AI - DASHBOARD FOUNDATION
# ==========================================

PROJECT_DIR = Path(
    r"C:\Users\adity\Downloads\Historical data\BharatMarket_AI"
)

DASHBOARD_DIR = PROJECT_DIR / "dashboard"
PAGES_DIR = PROJECT_DIR / "pages"

DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)
PAGES_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------
# 1. Create dashboard package
# ------------------------------------------

init_file = DASHBOARD_DIR / "__init__.py"

init_file.write_text(
    '"""BharatMarket AI dashboard package."""\n',
    encoding="utf-8"
)

# ------------------------------------------
# 2. Create Bloomberg-inspired theme
# ------------------------------------------

theme_code = r'''
import streamlit as st

BG = "#080D16"
PANEL = "#101927"
PANEL_LIGHT = "#152235"
BORDER = "#263449"
TEXT = "#E5EDF7"
MUTED = "#8797AB"
BLUE = "#42A5FF"
GREEN = "#16C784"
RED = "#EA3943"
AMBER = "#F5B942"


def inject_theme():

    st.markdown(
        """
        <style>

        @import url(
          'https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;500;600;700&family=Inter:wght@400;500;600;700;800&display=swap'
        );

        :root {
            --bg: #080D16;
            --panel: #101927;
            --panel-light: #152235;
            --border: #263449;
            --text: #E5EDF7;
            --muted: #8797AB;
            --blue: #42A5FF;
            --green: #16C784;
            --red: #EA3943;
            --amber: #F5B942;
        }

        html, body, [class*="css"] {
            font-family: 'Inter', sans-serif;
        }

        .stApp {
            background: var(--bg);
            color: var(--text);
        }

        [data-testid="stHeader"] {
            background: rgba(8,13,22,0.96);
        }

        [data-testid="stSidebar"] {
            background: #0B1320;
            border-right: 1px solid var(--border);
        }

        [data-testid="stSidebar"] > div {
            padding-top: 1rem;
        }

        [data-testid="stSidebar"] p {
            color: var(--text);
        }

        .block-container {
            padding-top: 1.5rem;
            padding-bottom: 2rem;
            max-width: 1800px;
        }

        h1, h2, h3 {
            color: var(--text) !important;
            letter-spacing: -0.4px;
        }

        .terminal-brand {
            font-family: 'IBM Plex Mono', monospace;
            font-weight: 700;
            font-size: 20px;
            color: #FFFFFF;
            letter-spacing: 1px;
        }

        .terminal-subtitle {
            color: var(--muted);
            font-family: 'IBM Plex Mono', monospace;
            font-size: 10px;
            letter-spacing: 1.8px;
        }

        .terminal-header {
            display: flex;
            align-items: center;
            justify-content: space-between;
            gap: 12px;
            padding: 12px 16px;
            background: var(--panel);
            border: 1px solid var(--border);
            border-radius: 7px;
            margin-bottom: 22px;
        }

        .status-online {
            color: var(--green);
            font-family: 'IBM Plex Mono', monospace;
            font-size: 11px;
            font-weight: 600;
        }

        .status-offline {
            color: var(--amber);
            font-family: 'IBM Plex Mono', monospace;
            font-size: 11px;
            font-weight: 600;
        }

        .section-label {
            color: var(--muted);
            font-family: 'IBM Plex Mono', monospace;
            font-size: 11px;
            letter-spacing: 1.4px;
            margin-bottom: 10px;
        }

        .terminal-card {
            background: var(--panel);
            border: 1px solid var(--border);
            border-radius: 7px;
            padding: 16px;
            min-height: 112px;
        }

        .card-label {
            color: var(--muted);
            font-family: 'IBM Plex Mono', monospace;
            font-size: 11px;
            letter-spacing: 0.8px;
        }

        .card-value {
            color: var(--text);
            font-family: 'IBM Plex Mono', monospace;
            font-size: 25px;
            font-weight: 600;
            margin-top: 12px;
        }

        .card-note {
            color: var(--muted);
            font-size: 11px;
            margin-top: 6px;
        }

        .terminal-divider {
            border-top: 1px solid var(--border);
            margin: 20px 0;
        }

        .mono {
            font-family: 'IBM Plex Mono', monospace;
        }

        .stButton button {
            background: #14243A;
            color: var(--text);
            border: 1px solid var(--border);
            border-radius: 5px;
        }

        .stButton button:hover {
            border-color: var(--blue);
            color: var(--blue);
        }

        [data-testid="stMetric"] {
            background: var(--panel);
            border: 1px solid var(--border);
            border-radius: 7px;
            padding: 13px;
        }

        [data-testid="stMetricLabel"] {
            color: var(--muted);
        }

        [data-testid="stDataFrame"] {
            border: 1px solid var(--border);
            border-radius: 7px;
        }

        [data-testid="stAlert"] {
            border-radius: 6px;
        }

        footer {
            visibility: hidden;
        }

        </style>
        """,
        unsafe_allow_html=True
    )


def terminal_card(label, value, note="AWAITING DATA"):

    st.markdown(
        f"""
        <div class="terminal-card">
            <div class="card-label">{label}</div>
            <div class="card-value">{value}</div>
            <div class="card-note">{note}</div>
        </div>
        """,
        unsafe_allow_html=True
    )
'''

theme_file = DASHBOARD_DIR / "theme.py"
theme_file.write_text(theme_code, encoding="utf-8")

# ------------------------------------------
# 3. Create main application
# ------------------------------------------

app_code = r'''
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

import streamlit as st

from dashboard.theme import inject_theme, terminal_card

# ==========================================
# APPLICATION CONFIGURATION
# ==========================================

PROJECT_DIR = Path(__file__).resolve().parent
IST = ZoneInfo("Asia/Kolkata")

st.set_page_config(
    page_title="BharatMarket AI | Intelligence Terminal",
    page_icon="📈",
    layout="wide",
    initial_sidebar_state="expanded"
)

inject_theme()

# ==========================================
# NAVIGATION
# ==========================================

NAVIGATION = {
    "MARKET TERMINAL": [
        "Overview",
        "Market Analysis",
        "Live Market Feed",
        "Trading Signals",
    ],
    "AI INTELLIGENCE": [
        "AI Forecasting",
        "Backtesting",
        "Risk Analytics",
    ],
    "DATA & SYSTEM": [
        "Data Explorer",
        "System Monitor",
    ],
}

# ==========================================
# SIDEBAR
# ==========================================

with st.sidebar:

    st.markdown(
        """
        <div style="padding:8px 0 20px 0">
            <div style="font-size:25px">📈</div>
            <div class="terminal-brand">
                BHARATMARKET
            </div>
            <div class="terminal-subtitle">
                AI INTELLIGENCE TERMINAL
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )

    st.markdown("---")

    st.markdown(
        '<div class="section-label">WORKSPACE</div>',
        unsafe_allow_html=True
    )

    selected_page = None

    for section, pages in NAVIGATION.items():

        st.markdown(
            f'<div class="section-label">{section}</div>',
            unsafe_allow_html=True
        )

        choice = st.radio(
            section,
            pages,
            key=f"navigation_{section}",
            label_visibility="collapsed"
        )

        if choice:
            selected_page = choice

    st.markdown("---")

    st.markdown(
        """
        <div class="section-label">PLATFORM INFORMATION</div>
        <div style="font-size:12px;line-height:2">
            <div>PROJECT <b>BM-AI-001</b></div>
            <div>MARKET <b>NSE INDIA</b></div>
            <div>MODE <b>PAPER TRADING</b></div>
            <div>VERSION <b>1.0.0</b></div>
        </div>
        """,
        unsafe_allow_html=True
    )

    st.markdown("---")

    st.caption("BHARATMARKET AI · ACADEMIC PROJECT")

# ==========================================
# TOP TERMINAL HEADER
# ==========================================

now = datetime.now(IST)

st.markdown(
    f"""
    <div class="terminal-header">
        <div>
            <div class="terminal-subtitle">
                INDIA EQUITY MARKET
            </div>
            <div style="font-size:17px;font-weight:700">
                BHARATMARKET AI
            </div>
        </div>

        <div style="text-align:right">
            <div class="status-offline">
                ● MARKET FEED NOT CONNECTED
            </div>
            <div class="terminal-subtitle">
                {now.strftime("%d %b %Y")} |
                {now.strftime("%H:%M:%S")} IST
            </div>
        </div>
    </div>
    """,
    unsafe_allow_html=True
)

# ==========================================
# PAGE HEADER
# ==========================================

st.markdown(
    f"""
    <div class="section-label">
        BHARATMARKET AI / WORKSPACE / {selected_page.upper()}
    </div>
    """,
    unsafe_allow_html=True
)

# ==========================================
# OVERVIEW PAGE
# ==========================================

if selected_page == "Overview":

    st.title("Market Overview")

    st.caption(
        "Indian equity market intelligence, "
        "AI forecasting and portfolio analytics."
    )

    st.markdown(
        '<div class="section-label">MARKET SNAPSHOT</div>',
        unsafe_allow_html=True
    )

    cols = st.columns(4)

    with cols[0]:
        terminal_card(
            "NIFTY 50",
            "—",
            "LIVE DATA NOT CONNECTED"
        )

    with cols[1]:
        terminal_card(
            "BANK NIFTY",
            "—",
            "LIVE DATA NOT CONNECTED"
        )

    with cols[2]:
        terminal_card(
            "ACTIVE INSTRUMENTS",
            "10",
            "PROJECT WATCHLIST"
        )

    with cols[3]:
        terminal_card(
            "AI SIGNAL",
            "—",
            "AWAITING MODEL INFERENCE"
        )

    st.markdown(
        '<div class="terminal-divider"></div>',
        unsafe_allow_html=True
    )

    left, right = st.columns([1.7, 1])

    with left:

        st.markdown(
            '<div class="section-label">MARKET WATCH</div>',
            unsafe_allow_html=True
        )

        watchlist = [
            "NIFTY50",
            "BANKNIFTY",
            "RELIANCE",
            "TCS",
            "HDFCBANK",
            "INFY",
            "ICICIBANK",
            "SBIN",
            "ITC",
            "LT",
        ]

        import pandas as pd

        watch_df = pd.DataFrame({
            "INSTRUMENT": watchlist,
            "LAST PRICE": ["—"] * 10,
            "CHANGE": ["—"] * 10,
            "AI VIEW": ["PENDING"] * 10,
            "DATA STATUS": ["NOT CONNECTED"] * 10,
        })

        st.dataframe(
            watch_df,
            use_container_width=True,
            hide_index=True
        )

    with right:

        st.markdown(
            '<div class="section-label">SYSTEM STATUS</div>',
            unsafe_allow_html=True
        )

        st.markdown(
            """
            <div class="terminal-card">
                <div class="card-label">UPSTOX MARKET DATA V3</div>
                <div class="card-value"
                     style="font-size:17px;color:#F5B942">
                    NOT CONNECTED
                </div>
                <div class="card-note">
                    LIVE CONNECTION NOT STARTED
                </div>
            </div>
            <br>
            <div class="terminal-card">
                <div class="card-label">AI MODEL ENGINE</div>
                <div class="card-value"
                     style="font-size:17px;color:#42A5FF">
                    NOT INITIALIZED
                </div>
                <div class="card-note">
                    MODEL REGISTRY CHECK PENDING
                </div>
            </div>
            <br>
            <div class="terminal-card">
                <div class="card-label">DATA PIPELINE</div>
                <div class="card-value"
                     style="font-size:17px;color:#42A5FF">
                    NOT INITIALIZED
                </div>
                <div class="card-note">
                    DATA SOURCE VALIDATION PENDING
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )

    st.markdown(
        '<div class="terminal-divider"></div>',
        unsafe_allow_html=True
    )

    st.markdown(
        '<div class="section-label">PROJECT MODULES</div>',
        unsafe_allow_html=True
    )

    modules = st.columns(3)

    module_data = [
        ("AI FORECASTING", "ML models and return predictions"),
        ("BACKTESTING", "Strategy performance and trade history"),
        ("RISK ANALYTICS", "Portfolio risk and drawdown metrics"),
    ]

    for col, (name, description) in zip(modules, module_data):

        with col:

            st.markdown(
                f"""
                <div class="terminal-card">
                    <div class="card-label">{name}</div>
                    <div style="font-size:13px;margin-top:12px">
                        {description}
                    </div>
                    <div class="card-note">
                        MODULE AVAILABLE IN NAVIGATION
                    </div>
                </div>
                """,
                unsafe_allow_html=True
            )

# ==========================================
# OTHER PAGE PLACEHOLDERS
# ==========================================

else:

    page_descriptions = {
        "Market Analysis":
            "Interactive charts, historical prices and technical indicators.",
        "Live Market Feed":
            "Upstox WebSocket market data and streaming feed monitor.",
        "Trading Signals":
            "AI-assisted market signals, entry, targets and stop loss.",
        "AI Forecasting":
            "Machine-learning forecasts and model evaluation.",
        "Backtesting":
            "Strategy returns, trade history and benchmark comparison.",
        "Risk Analytics":
            "Portfolio volatility, VaR, CVaR, Sharpe and drawdown.",
        "Data Explorer":
            "Historical datasets, feature data and data quality.",
        "System Monitor":
            "Model registry, API health, pipeline logs and diagnostics.",
    }

    st.title(selected_page)

    st.caption(page_descriptions.get(selected_page, ""))

    st.info(
        "This module is part of the BharatMarket AI implementation plan. "
        "Its existing data and model integrations will be added in the "
        "next development stages."
    )

    st.markdown(
        """
        <div class="terminal-card">
            <div class="card-label">MODULE STATUS</div>
            <div class="card-value" style="font-size:19px;color:#F5B942">
                FOUNDATION CREATED
            </div>
            <div class="card-note">
                DATA INTEGRATION AND FUNCTIONALITY PENDING
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )

# ==========================================
# FOOTER
# ==========================================

st.markdown(
    '<div class="terminal-divider"></div>',
    unsafe_allow_html=True
)

st.markdown(
    """
    <div style="
        display:flex;
        justify-content:space-between;
        color:#718096;
        font-size:10px;
        font-family:'IBM Plex Mono',monospace;
    ">
        <span>BHARATMARKET AI · FINANCIAL INTELLIGENCE</span>
        <span>ACADEMIC PROJECT · PAPER TRADING ONLY</span>
    </div>
    """,
    unsafe_allow_html=True
)
'''

app_file = PROJECT_DIR / "app.py"

# Back up an existing app.py instead of silently destroying it
if app_file.exists():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_file = PROJECT_DIR / f"app_backup_{timestamp}.py"
    shutil.copy2(app_file, backup_file)
    print("Existing app.py backed up to:", backup_file)

app_file.write_text(app_code, encoding="utf-8")

# ------------------------------------------
# 4. Verify created files
# ------------------------------------------

print("\n" + "=" * 55)
print("BHARATMARKET AI DASHBOARD FOUNDATION CREATED")
print("=" * 55)

for file in [
    app_file,
    init_file,
    theme_file
]:
    print(("OK" if file.exists() else "MISSING"), file)

print("\nProject directory:", PROJECT_DIR)
print("Dashboard directory:", DASHBOARD_DIR)
print("Pages directory:", PAGES_DIR)



BHARATMARKET AI DASHBOARD FOUNDATION CREATED
OK C:\Users\adity\Downloads\Historical data\BharatMarket_AI\app.py
OK C:\Users\adity\Downloads\Historical data\BharatMarket_AI\dashboard\__init__.py
OK C:\Users\adity\Downloads\Historical data\BharatMarket_AI\dashboard\theme.py

Project directory: C:\Users\adity\Downloads\Historical data\BharatMarket_AI
Dashboard directory: C:\Users\adity\Downloads\Historical data\BharatMarket_AI\dashboard
Pages directory: C:\Users\adity\Downloads\Historical data\BharatMarket_AI\pages


In [21]:

from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\adity\Downloads\Historical data\BharatMarket_AI"
)

DASHBOARD_DIR = PROJECT_DIR / "dashboard"
DASHBOARD_DIR.mkdir(exist_ok=True)

market_data_code = r'''
from pathlib import Path
import pandas as pd
import numpy as np

SYMBOLS = [
    "NIFTY50", "BANKNIFTY", "RELIANCE", "TCS",
    "HDFCBANK", "INFY", "ICICIBANK", "SBIN",
    "ITC", "LT"
]

STOCKS = [
    "RELIANCE", "TCS", "HDFCBANK", "INFY",
    "ICICIBANK", "SBIN", "ITC", "LT"
]


def load_instrument(project_dir, symbol):

    file_path = (
        Path(project_dir) / "data" / "raw" / f"{symbol}.csv"
    )

    if not file_path.exists():
        raise FileNotFoundError(
            f"Historical data file not found: {file_path}"
        )

    df = pd.read_csv(file_path)

    df.columns = [
        str(c).strip().lower() for c in df.columns
    ]

    required = [
        "date", "open", "high", "low", "close", "volume"
    ]

    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(
            f"{symbol}: Missing columns {missing}"
        )

    df["date"] = pd.to_datetime(
        df["date"], errors="coerce"
    )

    numeric_columns = [
        "open", "high", "low", "close", "volume"
    ]

    for col in numeric_columns:
        df[col] = pd.to_numeric(
            df[col], errors="coerce"
        )

    df = (
        df.dropna(subset=["date", "close"])
        .sort_values("date")
        .drop_duplicates(subset=["date"], keep="last")
        .reset_index(drop=True)
    )

    df["symbol"] = symbol

    return df


def load_all_market_data(project_dir):

    data = {}
    errors = {}

    for symbol in SYMBOLS:
        try:
            data[symbol] = load_instrument(
                project_dir, symbol
            )
        except Exception as exc:
            errors[symbol] = str(exc)

    return data, errors


def add_indicators(df):

    df = df.copy()

    df["daily_return"] = df["close"].pct_change() * 100

    df["ema_20"] = df["close"].ewm(
        span=20, adjust=False
    ).mean()

    df["ema_50"] = df["close"].ewm(
        span=50, adjust=False
    ).mean()

    delta = df["close"].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(
        alpha=1/14, adjust=False, min_periods=14
    ).mean()

    avg_loss = loss.ewm(
        alpha=1/14, adjust=False, min_periods=14
    ).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)

    df["rsi_14"] = 100 - (100 / (1 + rs))

    return df


def get_instrument_summary(df):

    if df.empty:
        return {}

    latest = df.iloc[-1]

    previous_close = (
        df.iloc[-2]["close"] if len(df) >= 2 else np.nan
    )

    change = (
        latest["close"] - previous_close
        if pd.notna(previous_close) else np.nan
    )

    change_pct = (
        change / previous_close * 100
        if pd.notna(previous_close)
        and previous_close != 0
        else np.nan
    )

    return {
        "date": latest["date"],
        "close": latest["close"],
        "previous_close": previous_close,
        "change": change,
        "change_pct": change_pct,
        "volume": latest["volume"],
        "open": latest["open"],
        "high": latest["high"],
        "low": latest["low"],
    }


def calculate_market_breadth(data):

    advances = 0
    declines = 0
    unchanged = 0
    unavailable = 0

    for symbol in STOCKS:

        df = data.get(symbol)

        if df is None or len(df) < 2:
            unavailable += 1
            continue

        change = df.iloc[-1]["close"] - df.iloc[-2]["close"]

        if change > 0:
            advances += 1
        elif change < 0:
            declines += 1
        else:
            unchanged += 1

    return {
        "advances": advances,
        "declines": declines,
        "unchanged": unchanged,
        "unavailable": unavailable,
        "total": len(STOCKS),
    }
'''

market_data_file = DASHBOARD_DIR / "market_data.py"
market_data_file.write_text(
    market_data_code,
    encoding="utf-8"
)

print("MARKET DATA ENGINE CREATED")
print(market_data_file)


MARKET DATA ENGINE CREATED
C:\Users\adity\Downloads\Historical data\BharatMarket_AI\dashboard\market_data.py


In [22]:

overview_code = r'''
from pathlib import Path

import streamlit as st
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from dashboard.market_data import (
    SYMBOLS,
    STOCKS,
    load_all_market_data,
    add_indicators,
    get_instrument_summary,
    calculate_market_breadth,
)


def render_overview(project_dir):

    st.title("Market Overview")

    st.caption(
        "Indian equity market intelligence | "
        "Historical market data and analytics"
    )

    data, errors = load_all_market_data(project_dir)

    if not data:
        st.error("No historical market datasets could be loaded.")

        for symbol, error in errors.items():
            st.write(f"{symbol}: {error}")

        return

    index_col1, index_col2 = st.columns(2)

    for col, symbol in zip(
        [index_col1, index_col2],
        ["NIFTY50", "BANKNIFTY"]
    ):

        with col:

            df = data.get(symbol)

            if df is None or df.empty:
                st.warning(f"{symbol}: Data unavailable")
                continue

            summary = get_instrument_summary(df)

            st.metric(
                label=symbol,
                value=f"{summary['close']:,.2f}",
                delta=(
                    f"{summary['change_pct']:+.2f}%"
                    if pd.notna(summary["change_pct"])
                    else None
                ),
                help=(
                    "Historical closing price. "
                    "Not a live market quote."
                )
            )

            st.caption(
                "Latest historical session: "
                + summary["date"].strftime("%d %b %Y")
            )

    st.divider()

    st.subheader("Market Breadth")

    breadth = calculate_market_breadth(data)

    b1, b2, b3, b4 = st.columns(4)

    b1.metric("Advances", breadth["advances"])
    b2.metric("Declines", breadth["declines"])
    b3.metric("Unchanged", breadth["unchanged"])
    b4.metric("Unavailable", breadth["unavailable"])

    st.caption(
        "Breadth is calculated using the eight equity stocks "
        "in the project watchlist, excluding the two indices."
    )

    st.divider()

    st.subheader("Historical Market Chart")

    available_symbols = [
        symbol for symbol in SYMBOLS if symbol in data
    ]

    selected_symbol = st.selectbox(
        "Select instrument",
        available_symbols,
        index=0
    )

    period = st.selectbox(
        "Historical period",
        ["1M", "3M", "6M", "1Y", "5Y"],
        index=2
    )

    df = add_indicators(data[selected_symbol])

    days = {
        "1M": 31,
        "3M": 92,
        "6M": 183,
        "1Y": 365,
        "5Y": 1826,
    }[period]

    latest_date = df["date"].max()
    cutoff = latest_date - pd.Timedelta(days=days)

    chart_df = df[df["date"] >= cutoff].copy()

    if chart_df.empty:
        st.warning("No observations available for this period.")
    else:

        fig = make_subplots(
            rows=2,
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.06,
            row_heights=[0.72, 0.28],
        )

        fig.add_trace(
            go.Candlestick(
                x=chart_df["date"],
                open=chart_df["open"],
                high=chart_df["high"],
                low=chart_df["low"],
                close=chart_df["close"],
                name="Price",
                increasing_line_color="#16C784",
                decreasing_line_color="#EA3943",
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=chart_df["date"],
                y=chart_df["ema_20"],
                name="EMA 20",
                line=dict(color="#42A5FF", width=1.4),
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=chart_df["date"],
                y=chart_df["ema_50"],
                name="EMA 50",
                line=dict(color="#F5B942", width=1.4),
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Bar(
                x=chart_df["date"],
                y=chart_df["volume"],
                name="Volume",
                marker_color="#52799D",
                opacity=0.65,
            ),
            row=2,
            col=1,
        )

        fig.update_layout(
            height=570,
            template="plotly_dark",
            paper_bgcolor="#101927",
            plot_bgcolor="#101927",
            font=dict(
                family="IBM Plex Mono, monospace",
                color="#DCE6F2",
                size=11,
            ),
            margin=dict(l=15, r=15, t=35, b=15),
            xaxis_rangeslider_visible=False,
            legend=dict(
                orientation="h",
                y=1.04,
                x=0,
            ),
            hovermode="x unified",
        )

        fig.update_xaxes(
            gridcolor="#263449",
            showgrid=True,
        )

        fig.update_yaxes(
            gridcolor="#263449",
            showgrid=True,
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

        latest = chart_df.iloc[-1]

        rsi_value = latest["rsi_14"]

        st.caption(
            f"Latest RSI (14): "
            f"{rsi_value:.2f}"
            if pd.notna(rsi_value)
            else "Latest RSI (14): unavailable"
        )

    st.divider()

    st.subheader("Instrument Watchlist")

    rows = []

    for symbol in SYMBOLS:

        if symbol not in data:
            rows.append({
                "Instrument": symbol,
                "Latest Date": "Unavailable",
                "Close": None,
                "Daily Change %": None,
                "Volume": None,
                "Status": "DATA MISSING",
            })
            continue

        summary = get_instrument_summary(data[symbol])

        rows.append({
            "Instrument": symbol,
            "Latest Date": summary["date"].strftime("%d %b %Y"),
            "Close": summary["close"],
            "Daily Change %": summary["change_pct"],
            "Volume": summary["volume"],
            "Status": "HISTORICAL DATA",
        })

    watchlist_df = pd.DataFrame(rows)

    st.dataframe(
        watchlist_df,
        use_container_width=True,
        hide_index=True,
        column_config={
            "Close": st.column_config.NumberColumn(
                "Close",
                format="%.2f"
            ),
            "Daily Change %": st.column_config.NumberColumn(
                "Daily Change %",
                format="%.2f%%"
            ),
            "Volume": st.column_config.NumberColumn(
                "Volume",
                format="localized"
            ),
        }
    )

    st.divider()

    st.subheader("Data Quality and Availability")

    q1, q2, q3 = st.columns(3)

    q1.metric(
        "Loaded instruments",
        len(data)
    )

    q2.metric(
        "Missing instruments",
        len(errors)
    )

    dates = [
        df["date"].max()
        for df in data.values()
        if not df.empty
    ]

    latest_common = min(dates) if dates else None

    q3.metric(
        "Latest available date",
        latest_common.strftime("%d %b %Y")
        if latest_common is not None
        else "Unavailable"
    )

    if errors:

        with st.expander("Dataset loading errors"):
            for symbol, error in errors.items():
                st.error(f"{symbol}: {error}")

    st.info(
        "DATA CLASSIFICATION: HISTORICAL DAILY CSV DATA. "
        "The displayed values are not live quotes. "
        "Upstox live-feed integration will be added separately."
    )
'''

overview_file = DASHBOARD_DIR / "overview.py"
overview_file.write_text(overview_code, encoding="utf-8")

print("OVERVIEW MODULE CREATED")
print(overview_file)


OVERVIEW MODULE CREATED
C:\Users\adity\Downloads\Historical data\BharatMarket_AI\dashboard\overview.py


In [23]:

from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

import streamlit as st

from dashboard.theme import inject_theme
from dashboard.overview import render_overview

PROJECT_DIR = Path(__file__).resolve().parent
IST = ZoneInfo("Asia/Kolkata")

st.set_page_config(
    page_title="BharatMarket AI | Intelligence Terminal",
    page_icon="📈",
    layout="wide",
    initial_sidebar_state="expanded",
)

inject_theme()

NAVIGATION = {
    "MARKET TERMINAL": [
        "Overview",
        "Market Analysis",
        "Live Market Feed",
        "Trading Signals",
    ],
    "AI INTELLIGENCE": [
        "AI Forecasting",
        "Backtesting",
        "Risk Analytics",
    ],
    "DATA & SYSTEM": [
        "Data Explorer",
        "System Monitor",
    ],
}

with st.sidebar:

    st.markdown(
        """
        <div style="padding:8px 0 20px 0">
            <div style="font-size:25px">📈</div>
            <div class="terminal-brand">BHARATMARKET</div>
            <div class="terminal-subtitle">
                AI INTELLIGENCE TERMINAL
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )

    st.markdown("---")

    selected_page = "Overview"

    for section, pages in NAVIGATION.items():

        st.markdown(
            f'<div class="section-label">{section}</div>',
            unsafe_allow_html=True,
        )

        choice = st.radio(
            section,
            pages,
            key=f"navigation_{section}",
            label_visibility="collapsed",
        )

        if choice:
            if choice in pages:
                # The selected page is the active navigation item
                pass

        # Store the selection from each navigation group
        if choice:
            if st.session_state.get("active_page") is None:
                st.session_state["active_page"] = "Overview"

            if choice != "Overview":
                if st.session_state.get(
                    f"navigation_{section}"
                ) == choice:
                    pass

    # Determine active selection from radio states
    for section in NAVIGATION:
        value = st.session_state.get(
            f"navigation_{section}"
        )

        if value:
            selected_page = value

    st.session_state["active_page"] = selected_page

    st.markdown("---")

    st.markdown(
        """
        <div class="section-label">PLATFORM INFORMATION</div>
        <div style="font-size:12px;line-height:2">
            <div>PROJECT <b>BM-AI-001</b></div>
            <div>MARKET <b>NSE INDIA</b></div>
            <div>MODE <b>PAPER TRADING</b></div>
            <div>VERSION <b>1.0.0</b></div>
        </div>
        """,
        unsafe_allow_html=True,
    )

now = datetime.now(IST)

st.markdown(
    f"""
    <div class="terminal-header">
        <div>
            <div class="terminal-subtitle">
                INDIA EQUITY MARKET
            </div>
            <div style="font-size:17px;font-weight:700">
                BHARATMARKET AI
            </div>
        </div>
        <div style="text-align:right">
            <div class="status-offline">
                ● HISTORICAL DATA MODE
            </div>
            <div class="terminal-subtitle">
                {now.strftime("%d %b %Y")} |
                {now.strftime("%H:%M:%S")} IST
            </div>
        </div>
    </div>
    """,
    unsafe_allow_html=True,
)

if selected_page == "Overview":

    render_overview(PROJECT_DIR)

else:

    descriptions = {
        "Market Analysis":
            "Interactive market charts and technical analysis.",
        "Live Market Feed":
            "Upstox V3 WebSocket streaming and market updates.",
        "Trading Signals":
            "Model-driven trading signals and risk/reward analysis.",
        "AI Forecasting":
            "Machine-learning forecasts and model evaluation.",
        "Backtesting":
            "Strategy performance, trade history and benchmarks.",
        "Risk Analytics":
            "Portfolio risk, VaR, CVaR and drawdown.",
        "Data Explorer":
            "Historical datasets and feature data.",
        "System Monitor":
            "Model registry, APIs and pipeline diagnostics.",
    }

    st.title(selected_page)
    st.caption(descriptions.get(selected_page, ""))

    st.info(
        "Module foundation created. "
        "The functional implementation will be connected "
        "to your existing project artifacts in the next stages."
    )

st.markdown(
    '<div class="terminal-divider"></div>',
    unsafe_allow_html=True,
)

st.caption(
    "BHARATMARKET AI | FINANCIAL INTELLIGENCE | "
    "ACADEMIC PROJECT | PAPER TRADING ONLY"
)


NameError: name '__file__' is not defined

In [29]:

result = service.start()
print(result)


{'status': 'starting', 'connected': False}
[UPSTOX] WebSocket connected


In [30]:

import time

time.sleep(5)

snapshot = service.get_snapshot()

print("CONNECTION STATUS:")
print(snapshot["status"])

print("\nLATEST MARKET DATA:")
display(snapshot["data"])


CONNECTION STATUS:
{'connected': True, 'started': True, 'last_error': None, 'last_message_time': '2026-11-25T21:59:17.443123+05:30', 'message_count': 2, 'update_count': 0, 'tick_file': 'C:\\Users\\adity\\Downloads\\Historical data\\BharatMarket_AI\\upstox_integration\\live\\ticks\\normalized_ticks_20261125_215906.csv'}

LATEST MARKET DATA:


,symbol,instrument_key,ltp,exchange_timestamp_ms,cumulative_volume,previous_close,last_trade_quantity,message_kind,received_at,exchange_time,change,change_pct
0,ITC,NSE_EQ|INE154A01025,309.25,1795602573025,41071647.0,301.00,4,initial_snapshot,2026-11-25T21:59:17.443123+05:30,2026-11-25 15:59:33.025+05:30,8.25,2.740864
1,SBIN,NSE_EQ|INE062A01020,1138.15,1795602596981,11608572.0,1132.65,20,initial_snapshot,2026-11-25T21:59:17.444137+05:30,2026-11-25 15:59:56.981+05:30,5.50,0.485587
2,RELIANCE,NSE_EQ|INE002A01018,1373.90,1795602589560,18781717.0,1351.00,1,initial_snapshot,2026-11-25T21:59:17.445131+05:30,2026-11-25 15:59:49.560+05:30,22.90,1.695041
3,HDFCBANK,NSE_EQ|INE040A01034,846.20,1795602558752,48187313.0,817.80,10,initial_snapshot,2026-11-25T21:59:17.446131+05:30,2026-11-25 15:59:18.752+05:30,28.40,3.472732
4,INFY,NSE_EQ|INE009A01021,1130.00,1795602592059,14498633.0,1103.40,1,initial_snapshot,2026-11-25T21:59:17.447137+05:30,2026-11-25 15:59:52.059+05:30,26.60,2.410730
5,TCS,NSE_EQ|INE467B01029,2517.30,1795602553519,5001661.0,2460.00,145,initial_snapshot,2026-11-25T21:59:17.447137+05:30,2026-11-25 15:59:13.519+05:30,57.30,2.329268
6,BANKNIFTY,NSE_INDEX|Nifty Bank,63703.25,1795602600000,NaN,62585.35,NaN,initial_snapshot,2026-11-25T21:59:17.449129+05:30,2026-11-25 16:00:00.000+05:30,1117.90,1.786201
7,LT,NSE_EQ|INE018A01030,4156.00,1795602509247,1313101.0,4099.90,1,initial_snapshot,2026-11-25T21:59:17.449129+05:30,2026-11-25 15:58:29.247+05:30,56.10,1.368326
8,NIFTY50,NSE_INDEX|Nifty 50,25819.25,1795602600000,NaN,25349.65,NaN,initial_snapshot,2026-11-25T21:59:17.450135+05:30,2026-11-25 16:00:00.000+05:30,469.60,1.852491
9,ICICIBANK,NSE_EQ|INE090A01021,1640.00,1795602555908,18982380.0,1585.50,50,initial_snapshot,2026-11-25T21:59:17.450135+05:30,2026-11-25 15:59:15.908+05:30,54.50,3.437401


In [37]:
service.stop()

[UPSTOX] Disconnect requested
